In [10]:
import pandas as pd

# 1. Load both datasets
covid = pd.read_csv("COVID.csv")
ed = pd.read_csv("ED.csv",low_memory=False)



In [11]:
covid.head()

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-01-05,NaN,0.0,NaN,NaN,0.0,NaN,...,NaN,37.746,0.5,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-01-06,NaN,0.0,NaN,NaN,0.0,NaN,...,NaN,37.746,0.5,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-01-07,NaN,0.0,NaN,NaN,0.0,NaN,...,NaN,37.746,0.5,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-01-08,NaN,0.0,NaN,NaN,0.0,NaN,...,NaN,37.746,0.5,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-01-09,NaN,0.0,NaN,NaN,0.0,NaN,...,NaN,37.746,0.5,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN


In [12]:
ed.head()

,COUNTRY,Country,WEEK,Week number,GENDER,Gender,AGE,Age,VARIABLE,Variable,YEAR,Year,Value,Flag Codes,Flags
0,CZE,Czechia,46,46,TOTAL,Total,Y0T44,0 to 44,EXCESSNB,Excess deaths (number),2020,2020,2.2,NaN,NaN
1,CZE,Czechia,46,46,TOTAL,Total,Y0T44,0 to 44,EXCESSNB,Excess deaths (number),2021,2021,8.2,NaN,NaN
2,CZE,Czechia,46,46,TOTAL,Total,Y0T44,0 to 44,EXCESSNB,Excess deaths (number),2022,2022,4.2,NaN,NaN
3,NLD,Netherlands,3,3,TOTAL,Total,Y_GE65,65 and over,EXCESSNB,Excess deaths (number),2020,2020,-127.4,NaN,NaN
4,NLD,Netherlands,3,3,TOTAL,Total,Y_GE65,65 and over,EXCESSNB,Excess deaths (number),2021,2021,569.6,NaN,NaN


In [13]:
# 2. Clean column names: lowercase, replace spaces, etc.
covid_weekly.columns = covid_weekly.columns.str.strip().str.lower().str.replace(" ", "_")
ed.columns = ed.columns.str.strip().str.lower().str.replace(" ", "_")

In [14]:
# 3. Remove duplicate columns from ED (important fix!)
ed = ed.loc[:, ~ed.columns.duplicated()]


In [15]:
# 4. Make sure 'country' column is renamed to 'location' for merging
if 'country' in ed.columns:
    ed.rename(columns={'country': 'location'}, inplace=True)

C:\Users\avich\AppData\Local\Temp\ipykernel_27912\2909998753.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ed.rename(columns={'country': 'location'}, inplace=True)


In [16]:
# 5. Make sure covid_weekly also uses 'location'
if 'country' in covid_weekly.columns:
    covid_weekly.rename(columns={'country': 'location'}, inplace=True)

In [17]:
# 6. Optional: ensure 'year' and 'week' are integers
for col in ['year', 'week']:
    if col in covid_weekly.columns:
        covid_weekly[col] = covid_weekly[col].astype(int, errors='ignore')
    if col in ed.columns:
        ed[col] = ed[col].astype(int, errors='ignore')


C:\Users\avich\AppData\Local\Temp\ipykernel_27912\3362525861.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ed[col] = ed[col].astype(int, errors='ignore')
C:\Users\avich\AppData\Local\Temp\ipykernel_27912\3362525861.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ed[col] = ed[col].astype(int, errors='ignore')


In [18]:
# 7. Inspect columns before merging (debugging tip)
print("COVID Weekly Columns:", covid_weekly.columns)
print("ED Columns:", ed.columns)


COVID Weekly Columns: Index(['location', 'year', 'week', 'total_cases', 'new_cases', 'total_deaths',
       'new_deaths', 'total_vaccinations', 'new_vaccinations', 'population',
       'gdp_per_capita', 'human_development_index'],
      dtype='object')
ED Columns: Index(['location', 'week', 'week_number', 'gender', 'age', 'variable', 'year',
       'value', 'flag_codes', 'flags'],
      dtype='object')


In [19]:
# 8. Select relevant columns from ED before merging
ed_reduced = ed[['location', 'year', 'week', 'age', 'gender', 'value']]


In [20]:

# 9. Merge datasets on location, year, and week
merged = pd.merge(
    covid_weekly,
    ed_reduced,
    on=['location', 'year', 'week'],
    how='outer'
)


In [21]:
# 10. Check the merged dataset
print("✅ Merged dataset shape:", merged.shape)
print("✅ Merged dataset preview:")
print(merged.head())


✅ Merged dataset shape: (169185, 15)
✅ Merged dataset preview:
  location  year  week  total_cases  new_cases  total_deaths  new_deaths  \
0      AUS  2020     1          NaN        NaN           NaN         NaN   
1      AUS  2020     1          NaN        NaN           NaN         NaN   
2      AUS  2020     1          NaN        NaN           NaN         NaN   
3      AUS  2020     1          NaN        NaN           NaN         NaN   
4      AUS  2020     1          NaN        NaN           NaN         NaN   

   total_vaccinations  new_vaccinations  population  gdp_per_capita  \
0                 NaN               NaN         NaN             NaN   
1                 NaN               NaN         NaN             NaN   
2                 NaN               NaN         NaN             NaN   
3                 NaN               NaN         NaN             NaN   
4                 NaN               NaN         NaN             NaN   

   human_development_index     age gender  value  
0 

In [22]:
# 11. Save merged dataset to a new CSV (optional)
merged.to_csv("merged_dataset.csv", index=False)
print("✅ Merged dataset saved as 'merged_dataset.csv'")

✅ Merged dataset saved as 'merged_dataset.csv'
